# Reproduce typed WL structural similarity and category LOTO

This notebook recomputes the primary (h=2) WL results directly from the 15 reference LPs, 101 original LPs, and 36 variant LPs. It includes the graph construction, cumulative WL features, cosine similarity, same-category matching, ordinary category LOTO, and the final summary table used in the reply letter.

LOTO removes only the references carrying the test formulation's category. For example, a Mixture test removes the eight Mixture references but keeps the two Others references. Source LP files are read only.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import math
from collections import Counter
from statistics import mean, median

import gurobipy as gp
import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "outputs").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "outputs").is_dir(), f"Could not locate repository root from {Path.cwd()}"
PACKAGE = ROOT / "outputs" / "wl_code_package_20261015"
SOURCE_AUDIT = PACKAGE / "results" / "audit.json"
OUTPUT = ROOT / "outputs" / "wl_ipynb_reproduction_20261018"
OUTPUT.mkdir(parents=True, exist_ok=True)
H = 2
CATEGORY_ORDER = ["NRM", "RA", "TP", "AP", "UFLP", "Mixture", "Others"]

print({"root": str(ROOT), "h": H, "output": str(OUTPUT)})

{'root': '/Users/cora/Documents/GitHub/lean-llm-opt', 'h': 2, 'output': '/Users/cora/Documents/GitHub/lean-llm-opt/outputs/wl_ipynb_reproduction_20261018'}


## Input manifest and scope

The saved package audit is used only as a manifest of LP paths and category labels. All similarity values are recomputed below from the LP files.

In [2]:
source_audit = json.loads(SOURCE_AUDIT.read_text(encoding="utf-8"))
records = source_audit["graph_audit"]
assert len(records) == 152
refs = [r for r in records if r["dataset"] == "reference"]
assert len(refs) == 15
assert Counter(r["dataset"] for r in records) == {
    "reference": 15, "large_scale_101": 101, "variants": 36
}

category_counts = Counter((r["dataset"], r["category"]) for r in records)
print("records:", len(records))
print("references:", [(r["instance"], r["category"]) for r in refs])
print("category counts:")
for key, value in sorted(category_counts.items()):
    print(f"  {key}: {value}")

records: 152
references: [('Ref01', 'AP'), ('Ref02', 'UFLP'), ('Ref03', 'NRM'), ('Ref04', 'Mixture'), ('Ref05', 'Mixture'), ('Ref06', 'Mixture'), ('Ref07', 'Mixture'), ('Ref08', 'Mixture'), ('Ref09', 'Mixture'), ('Ref10', 'Mixture'), ('Ref11', 'Mixture'), ('Ref12', 'Others'), ('Ref13', 'Others'), ('Ref14', 'RA'), ('Ref15', 'TP')]
category counts:
  ('large_scale_101', 'AP'): 5
  ('large_scale_101', 'Mixture'): 18
  ('large_scale_101', 'NRM'): 25
  ('large_scale_101', 'Others'): 8
  ('large_scale_101', 'RA'): 22
  ('large_scale_101', 'TP'): 9
  ('large_scale_101', 'UFLP'): 14
  ('reference', 'AP'): 1
  ('reference', 'Mixture'): 8
  ('reference', 'NRM'): 1
  ('reference', 'Others'): 2
  ('reference', 'RA'): 1
  ('reference', 'TP'): 1
  ('reference', 'UFLP'): 1
  ('variants', 'AP'): 3
  ('variants', 'Mixture'): 17
  ('variants', 'Others'): 13
  ('variants', 'UFLP'): 3


## Graph construction

For each LP, singleton rows are intersected with native variable bounds and omitted as graph nodes. The graph retains variable domains, non-singleton rows, isolated variables, and one edge per nonzero matrix coefficient. Labels are (V:B/V:I/V:C) for variables and (R:E/R:I) for rows.

In [3]:
def graph_from_lp(path, env):
    model = gp.read(str(path), env=env)
    try:
        if (model.NumQConstrs or model.NumQNZs or model.NumGenConstrs
                or model.NumSOS or model.NumPWLObjVars):
            raise ValueError(f"Only linear models are supported: {path}")
        variables = model.getVars()
        rows = model.getConstrs()
        types = [v.VType for v in variables]
        if not set(types).issubset({"B", "I", "C"}):
            raise ValueError(f"Unsupported variable domains: {path}")
        matrix = model.getA().tocsr()
        matrix.eliminate_zeros()
        bounds = [[float(v.LB), float(v.UB)] for v in variables]
        retained = []
        absorbed = []
        for i, row in enumerate(rows):
            start, stop = matrix.indptr[i:i + 2]
            if stop - start != 1:
                retained.append(i)
                continue
            j = int(matrix.indices[start])
            coefficient = float(matrix.data[start])
            value = float(row.RHS) / coefficient
            sense = row.Sense
            if coefficient < 0:
                sense = {"<": ">", ">": "<", "=": "="}[sense]
            if sense in (">", "="):
                bounds[j][0] = max(bounds[j][0], value)
            if sense in ("<", "="):
                bounds[j][1] = min(bounds[j][1], value)
            absorbed.append(row.ConstrName)

        normalized_bounds = []
        for j, (lo, hi) in enumerate(bounds):
            lo = lo if lo > -gp.GRB.INFINITY else -math.inf
            hi = hi if hi < gp.GRB.INFINITY else math.inf
            if types[j] in ("I", "B"):
                lo = math.ceil(lo) if math.isfinite(lo) else lo
                hi = math.floor(hi) if math.isfinite(hi) else hi
            normalized_bounds.append((lo, hi))

        n = len(variables)
        labels = ["V:" + t for t in types]
        labels += ["R:E" if rows[i].Sense == "=" else "R:I" for i in retained]
        adjacency = [[] for _ in labels]
        for k, i in enumerate(retained):
            start, stop = matrix.indptr[i:i + 2]
            for j in matrix.indices[start:stop]:
                j = int(j)
                adjacency[j].append(n + k)
                adjacency[n + k].append(j)
        stats = {
            "variables": n,
            "source_constraints": len(rows),
            "graph_constraints": len(retained),
            "graph_edges": sum(map(len, adjacency)) // 2,
            "absorbed_singleton_rows": len(absorbed),
            "isolated_variables": sum(not a for a in adjacency[:n]),
            "effective_bounds": normalized_bounds,
        }
        return (labels, adjacency), stats
    finally:
        model.dispose()

## Cumulative WL features and cosine kernels

In [4]:
def wl_kernels(graphs, max_h=2):
    labels = [label[:] for label, _ in graphs]
    accumulated = [Counter() for _ in graphs]
    kernels = {}
    for h in range(max_h + 1):
        for index, current in enumerate(labels):
            accumulated[index].update((h, label) for label in current)
        vocabulary = {}
        rows, cols, values = [], [], []
        for graph_index, counts in enumerate(accumulated):
            for key, count in counts.items():
                vocabulary.setdefault(key, len(vocabulary))
                rows.append(graph_index)
                cols.append(vocabulary[key])
                values.append(count)
        features = np.zeros((len(graphs), len(vocabulary)), dtype=float)
        for row, col, value in zip(rows, cols, values):
            features[row, col] = value
        raw = features @ features.T
        norms = np.sqrt(np.diag(raw))
        if np.any(norms == 0):
            raise ValueError("Cannot normalize an empty graph")
        kernels[h] = np.clip(raw / np.outer(norms, norms), 0.0, 1.0)
        if h == max_h:
            continue
        codebook = {}
        updated_all = []
        for current, (_, adjacency) in zip(labels, graphs):
            updated = []
            for vertex, neighbors in enumerate(adjacency):
                signature = (current[vertex], tuple(sorted(current[x] for x in neighbors)))
                codebook.setdefault(signature, len(codebook))
                updated.append(codebook[signature])
            updated_all.append(updated)
        labels = updated_all
    return kernels

## Recompute graphs and the (h=2) kernel

In [5]:
graphs = []
graph_stats = []
with gp.Env(empty=True) as env:
    env.setParam("OutputFlag", 0)
    env.start()
    for record in records:
        graph, stats = graph_from_lp(Path(record["lp_path"]), env)
        graphs.append(graph)
        graph_stats.append({
            "dataset": record["dataset"],
            "instance": record["instance"],
            "category": record["category"],
            **{k: v for k, v in stats.items() if k != "effective_bounds"},
        })

kernels = wl_kernels(graphs, max_h=H)
K = kernels[H]
assert K.shape == (152, 152)
assert np.allclose(K, K.T)
assert np.allclose(np.diag(K), 1.0)
print("kernel shape:", K.shape)
print("symmetry and unit diagonal: passed")

kernel shape: (152, 152)
symmetry and unit diagonal: passed


## Same-category matching and ordinary category LOTO

In [6]:
ref_indices = [i for i, r in enumerate(records) if r["dataset"] == "reference"]
refs = [records[i] for i in ref_indices]
assert len(refs) == 15

details = []
summaries = []
for dataset, expected_count, display_name in [
    ("large_scale_101", 101, "Original"),
    ("variants", 36, "Variant"),
]:
    test_indices = [i for i, r in enumerate(records) if r["dataset"] == dataset]
    assert len(test_indices) == expected_count
    dataset_rows = []
    for i in test_indices:
        record = records[i]
        scores = np.asarray([K[i, j] for j in ref_indices], dtype=float)
        same = [j for j, ref in enumerate(refs) if ref["category"] == record["category"]]
        loto = [j for j, ref in enumerate(refs) if ref["category"] != record["category"]]
        same_best = max(same, key=lambda j: scores[j])
        loto_best = max(loto, key=lambda j: scores[j])
        row = {
            "dataset": dataset,
            "set": display_name,
            "instance": record["instance"],
            "category": record["category"],
            "nearest_same": refs[same_best]["instance"],
            "similarity_same": float(scores[same_best]),
            "nearest_loto": refs[loto_best]["instance"],
            "nearest_loto_category": refs[loto_best]["category"],
            "similarity_loto": float(scores[loto_best]),
            "excluded_references": ";".join(refs[j]["instance"] for j in same),
            "remaining_reference_count": len(loto),
        }
        dataset_rows.append(row)
        details.append(row)

    for category in CATEGORY_ORDER + ["All"]:
        group = dataset_rows if category == "All" else [r for r in dataset_rows if r["category"] == category]
        if not group:
            continue
        summaries.append({
            "dataset": dataset,
            "set": display_name,
            "category": category,
            "tests": len(group),
            "median_same": median(r["similarity_same"] for r in group),
            "mean_same": mean(r["similarity_same"] for r in group),
            "median_loto": median(r["similarity_loto"] for r in group),
            "mean_loto": mean(r["similarity_loto"] for r in group),
        })

print("per-instance rows:", len(details))
print("summary rows:", len(summaries))

per-instance rows: 137
summary rows: 13


## Verify the values against the reply-letter table

In [7]:
expected = {
    ("large_scale_101", "NRM"): (25, 0.3234, 0.3234, 0.3330, 0.3330),
    ("large_scale_101", "RA"): (22, 0.6562, 0.6564, 0.6500, 0.6505),
    ("large_scale_101", "TP"): (9, 0.6240, 0.6314, 0.6327, 0.6369),
    ("large_scale_101", "AP"): (5, 0.0508, 0.0551, 0.4424, 0.4437),
    ("large_scale_101", "UFLP"): (14, 0.5503, 0.5557, 0.4446, 0.4493),
    ("large_scale_101", "Mixture"): (18, 0.3983, 0.4499, 0.4057, 0.4327),
    ("large_scale_101", "Others"): (8, 0.2584, 0.3025, 0.3615, 0.4070),
    ("large_scale_101", "All"): (101, 0.5065, 0.4632, 0.4424, 0.4745),
    ("variants", "AP"): (3, 0.3569, 0.3582, 0.2749, 0.2752),
    ("variants", "UFLP"): (3, 0.1981, 0.1972, 0.3078, 0.3079),
    ("variants", "Mixture"): (17, 0.6665, 0.6157, 0.2520, 0.3782),
    ("variants", "Others"): (13, 0.2373, 0.2993, 0.4281, 0.4442),
    ("variants", "All"): (36, 0.4325, 0.4451, 0.3127, 0.3876),
}
for row in summaries:
    key = (row["dataset"], row["category"])
    target = expected[key]
    got = (
        row["tests"],
        round(row["median_same"], 4),
        round(row["mean_same"], 4),
        round(row["median_loto"], 4),
        round(row["mean_loto"], 4),
    )
    assert got == target, (key, got, target)
print("all 13 category and pooled rows match the LaTeX table after rounding to 4 decimals")

all 13 category and pooled rows match the LaTeX table after rounding to 4 decimals


## Export the reproducible result directory

In [8]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# JSON and CSV exports keep the complete per-instance and category-level results.
(OUTPUT / "nearest_h2.json").write_text(json.dumps(details, indent=2), encoding="utf-8")
(OUTPUT / "graph_stats.json").write_text(json.dumps(graph_stats, indent=2), encoding="utf-8")
(OUTPUT / "category_summary_h2.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")

with (OUTPUT / "category_summary_h2.tsv").open("w", encoding="utf-8", newline="") as handle:
    fields = ["dataset", "set", "category", "tests", "median_same", "mean_same", "median_loto", "mean_loto"]
    writer = csv.DictWriter(handle, fieldnames=fields, delimiter="\t")
    writer.writeheader()
    writer.writerows(summaries)

latex_lines = [
    r"\begin{tabular}{@{}llrrrrr@{}}",
    r"\toprule",
    r"Set & Category & Tests & Median $S_{\rm same}$ & Mean $S_{\rm same}$ & Median $S_{\rm LOTO}$ & Mean $S_{\rm LOTO}$\\",
    r"\midrule",
]
for row in summaries:
    latex_lines.append(
        f"{row['set']} & {row['category']} & {row['tests']} & "
        f"{row['median_same']:.4f} & {row['mean_same']:.4f} & "
        f"{row['median_loto']:.4f} & {row['mean_loto']:.4f}\\\\"
    )
    if row["category"] == "All" and row["dataset"] == "large_scale_101":
        latex_lines.append(r"\midrule")
latex_lines += [r"\bottomrule", r"\end{tabular}"]
(OUTPUT / "category_summary_h2.tex").write_text("\n".join(latex_lines) + "\n", encoding="utf-8")

md_lines = [
    "# Typed WL structural similarity reproduction",
    "",
    "Primary setting: cumulative rounds 0--2; effective bounds retained as variable restrictions and excluded from the graph.",
    "LOTO removes only references carrying the test formulation's category.",
    "",
    "| Set | Category | Tests | Median same | Mean same | Median LOTO | Mean LOTO |",
    "|---|---|---:|---:|---:|---:|---:|",
]
for row in summaries:
    md_lines.append(
        f"| {row['set']} | {row['category']} | {row['tests']} | "
        f"{row['median_same']:.4f} | {row['mean_same']:.4f} | "
        f"{row['median_loto']:.4f} | {row['mean_loto']:.4f} |"
    )
(OUTPUT / "RESULTS.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")

input_hashes = {r["lp_path"]: sha256(r["lp_path"]) for r in records}
run_audit = {
    "method": "Typed variable-constraint bipartite cumulative WL subtree kernel",
    "h": H,
    "normalization": "Cosine similarity of cumulative label counts from rounds 0..2",
    "bounds": "Singleton rows absorbed into effective bounds; bounds excluded from graph",
    "ignored": ["objective", "coefficient magnitudes/signs", "RHS", "bound values", "names"],
    "reference_count": len(refs),
    "original_count": 101,
    "variant_count": 36,
    "input_hashes": input_hashes,
    "source_audit": str(SOURCE_AUDIT),
    "table_check": "passed",
}
(OUTPUT / "audit.json").write_text(json.dumps(run_audit, indent=2), encoding="utf-8")
(OUTPUT / "reproduction_check.json").write_text(json.dumps({
    "passed": True,
    "summary_rows_checked": len(summaries),
    "per_instance_rows": len(details),
    "table_rounding": 4,
}, indent=2), encoding="utf-8")

print("wrote:", OUTPUT)
print("files:", sorted(p.name for p in OUTPUT.iterdir()))

wrote: /Users/cora/Documents/GitHub/lean-llm-opt/outputs/wl_ipynb_reproduction_20261018
files: ['RESULTS.md', 'audit.json', 'category_summary_h2.json', 'category_summary_h2.tex', 'category_summary_h2.tsv', 'graph_stats.json', 'nearest_h2.json', 'reproduction_check.json']


## Final table check

The exported `category_summary_h2.tex` is the compact table fragment corresponding to the reply letter. `RESULTS.md`, `category_summary_h2.tsv`, and `category_summary_h2.json` provide human-readable and machine-readable copies of the same 13 summary rows.